In [1]:
import pandas as pd
import numpy as np

medals = pd.read_csv("../data/processed/olympic_1896_2024_extended.csv")

In [2]:
# Keep only Summer Olympics
summer_years = [
1896,1900,1904,1908,1912,
1920,1924,1928,1932,1936,
1948,1952,1956,1960,1964,
1968,1972,1976,1980,1984,
1988,1992,1996,2000,2004,
2008,2012,2016,2020,2024
]

summer = medals[medals["Year"].isin(summer_years)]

india = summer[summer["NOC"] == "IND"].copy()

india.head()

,Name,Sex,NOC,Year,Sport,Event,Medal
505,S. Abdul Hamid,M,IND,1928,Athletics,Athletics Men's 110 metres Hurdles,NaN
506,S. Abdul Hamid,M,IND,1928,Athletics,Athletics Men's 400 metres Hurdles,NaN
895,Shiny Kurisingal Abraham-Wilson,F,IND,1984,Athletics,Athletics Women's 800 metres,NaN
896,Shiny Kurisingal Abraham-Wilson,F,IND,1984,Athletics,Athletics Women's 4 x 400 metres Relay,NaN
897,Shiny Kurisingal Abraham-Wilson,F,IND,1988,Athletics,Athletics Women's 800 metres,NaN


In [3]:
india_medals = india[india["Medal"].notna()].copy()

india_sport_year = (
    india_medals
    .drop_duplicates(subset=["Year", "Sport", "Event"])
    .groupby(["Year", "Sport"])
    .size()
    .reset_index(name="Total")
)

In [4]:
all_sports = india["Sport"].unique()
all_years = sorted(india["Year"].unique())

full_grid = pd.MultiIndex.from_product(
    [all_years, all_sports],
    names=["Year", "Sport"]
).to_frame(index=False)

india_sport_year_full = full_grid.merge(
    india_sport_year,
    on=["Year", "Sport"],
    how="left"
)

india_sport_year_full["Total"] = india_sport_year_full["Total"].fillna(0)

india_sport_year_full = india_sport_year_full.sort_values(["Sport", "Year"])

In [5]:
india_sport_year_full["career_avg"] = (
    india_sport_year_full
    .groupby("Sport")["Total"]
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

india_sport_year_full["delta_last"] = (
    india_sport_year_full
    .groupby("Sport")["Total"]
    .diff()
)

india_sport_year_full[["career_avg", "delta_last"]] = \
    india_sport_year_full[["career_avg", "delta_last"]].fillna(0)

In [6]:
train = india_sport_year_full[india_sport_year_full["Year"] <= 2020]
test  = india_sport_year_full[india_sport_year_full["Year"] == 2024]

features = ["career_avg", "delta_last"]

X_train = train[features]
y_train = train["Total"]

X_test = test[features]
y_test = test["Total"]

In [7]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

pred_test = model.predict(X_test)

print("Layer 2 Test MAE:", round(mean_absolute_error(y_test, pred_test), 3))
print("Layer 2 Test R2 :", round(r2_score(y_test, pred_test), 3))

Layer 2 Test MAE: 0.196
Layer 2 Test R2 : 0.384


In [8]:
latest = india_sport_year_full[
    india_sport_year_full["Year"] == 2024
].copy()

latest["Predicted_Medals_2028"] = model.predict(latest[features])

# Clip to realistic bounds
latest["Predicted_Medals_2028"] = \
    latest["Predicted_Medals_2028"].clip(0, 2)

latest_sorted = latest[
    ["Sport", "career_avg", "delta_last", "Predicted_Medals_2028"]
].sort_values("Predicted_Medals_2028", ascending=False)

latest_sorted.head(15)

,Sport,career_avg,delta_last,Predicted_Medals_2028
640,Shooting,0.16,3.0,0.974430
632,Hockey,0.48,0.0,0.165728
635,Wrestling,0.28,-1.0,0.139005
627,Weightlifting,0.08,-1.0,0.032588
637,Badminton,0.12,-1.0,0.027758
642,Boxing,0.12,-1.0,0.027758
625,Athletics,0.12,0.0,0.021813
629,Water Polo,0.00,0.0,0.006070
626,Table Tennis,0.00,0.0,0.006070
638,Swimming,0.00,0.0,0.006070


In [9]:
print(
    "Total Predicted Medals 2028 (Layer 2):",
    round(latest["Predicted_Medals_2028"].sum(), 2)
)

Total Predicted Medals 2028 (Layer 2): 1.49


In [10]:
import json
with open("../models/model_meta.json") as f:
    meta = json.load(f)
layer1_total = meta["india_2028_base_prediction"]
print("Layer 1 total loaded:", layer1_total)

layer2_total = latest["Predicted_Medals_2028"].sum()

scaling_factor = layer1_total / layer2_total

latest["Scaled_Pred_2028"] = (
    latest["Predicted_Medals_2028"] * scaling_factor
)

latest["Scaled_Pred_2028"] = latest["Scaled_Pred_2028"].round(3)

print("Layer 2 Total (Before Scaling):", round(layer2_total,2))
print("Layer 2 Total (After Scaling):",
      round(latest["Scaled_Pred_2028"].sum(),2))

Layer 1 total loaded: 2.86
Layer 2 Total (Before Scaling): 1.49
Layer 2 Total (After Scaling): 2.87


In [11]:
# ==========================================
# SPORT SENSITIVITY MATRIX (UPGRADED)
# ==========================================

sensitivity_results = []

epsilon = 0.25  # stronger stability floor

for _, sport_row in latest.iterrows():
    
    sport = sport_row["Sport"]
    
    # Base prediction (raw model prediction)
    base_x = sport_row[features].values.reshape(1, -1)
    base_pred = model.predict(base_x)[0]
    
    # Boosted scenario (+1 delta_last)
    boost_x = base_x.copy()
    boost_x[0, 1] += 1   # index 1 = delta_last
    
    boosted_pred = model.predict(boost_x)[0]
    
    # Absolute gain
    raw_gain = boosted_pred - base_pred
    
    # Stabilized elasticity
    elasticity = (raw_gain / max(base_pred, epsilon)) * 100
    
    # Hybrid Impact Score (important)
    impact_score = raw_gain * base_pred
    
    sensitivity_results.append({
        "Sport": sport,
        "Base_Prediction": base_pred,
        "Medal_Gain": raw_gain,
        "Elasticity_Index": elasticity,
        "Impact_Score": impact_score,
        "Current_Strength": sport_row["career_avg"]
    })

sensitivity_df = pd.DataFrame(sensitivity_results)

# ==========================================
# CATEGORIZATION USING HYBRID LOGIC
# ==========================================

gain_q66 = sensitivity_df["Medal_Gain"].quantile(0.66)
elastic_q66 = sensitivity_df["Elasticity_Index"].quantile(0.66)

def categorize(row):
    
    if row["Medal_Gain"] > gain_q66 and row["Elasticity_Index"] > elastic_q66:
        return "🔥 High Impact & High Voltage"
    
    elif row["Medal_Gain"] > gain_q66:
        return "🏗 Structural Multiplier"
    
    elif row["Elasticity_Index"] > elastic_q66:
        return "⚡ High Voltage (Low Base)"
    
    else:
        return "🧊 Stable"

sensitivity_df["Response_Type"] = sensitivity_df.apply(categorize, axis=1)

# Sort by Impact Score (most policy relevant)
sensitivity_df = sensitivity_df.sort_values("Impact_Score", ascending=False)

print("\n--- 🧠 Sport Sensitivity Matrix (Upgraded) ---\n")

display(
    sensitivity_df[
        ["Sport",
         "Base_Prediction",
         "Medal_Gain",
         "Elasticity_Index",
         "Impact_Score",
         "Response_Type"]
    ].round(3)
)


--- 🧠 Sport Sensitivity Matrix (Upgraded) ---



,Sport,Base_Prediction,Medal_Gain,Elasticity_Index,Impact_Score,Response_Type
14,Hockey,0.166,0.953,381.047,0.158,🔥 High Impact & High Voltage
4,Athletics,0.022,0.953,381.047,0.021,🧊 Stable
0,Alpine Skiing,0.006,0.953,381.047,0.006,🧊 Stable
11,Football,0.006,0.953,381.047,0.006,🧊 Stable
22,Water Polo,0.006,0.953,381.047,0.006,🧊 Stable
20,Table Tennis,0.006,0.953,381.047,0.006,🧊 Stable
19,Swimming,0.006,0.953,381.047,0.006,🧊 Stable
17,Sailing,0.006,0.953,381.047,0.006,🧊 Stable
16,Rowing,0.006,0.953,381.047,0.006,🧊 Stable
15,Judo,0.006,0.953,381.047,0.006,🧊 Stable


In [12]:
# ===============================
# STRATEGIC ROI (STABLE VERSION)
# ===============================

roi_results = []

model_features = ["career_avg", "delta_last"]

epsilon = 0.25   # stability floor to prevent % explosion

for _, row in latest.iterrows():
    
    sport = row["Sport"]
    
    # Current prediction
    current_x = row[model_features].values.reshape(1, -1)
    current_pred = model.predict(current_x)[0]
    
    # Boosted prediction (+1 momentum)
    boosted_x = current_x.copy()
    boosted_x[0, 1] += 1   # delta_last boost
    boosted_pred = model.predict(boosted_x)[0]
    
    # Absolute marginal gain
    marginal_gain = boosted_pred - current_pred
    
    # Stabilized ROI percentage
    stable_base = max(current_pred, epsilon)
    roi_percentage = (marginal_gain / stable_base) * 100
    
    roi_results.append({
        "Sport": sport,
        "Base_Prediction": current_pred,
        "Raw_Gain": marginal_gain,
        "ROI_Percentage": roi_percentage,
        "Structural_Strength": row["career_avg"]
    })

roi_df = pd.DataFrame(roi_results)

# Sort primarily by Raw Gain (policy relevant)
roi_df = roi_df.sort_values("Raw_Gain", ascending=False)

# ===============================
# STRATEGIC CATEGORIZATION
# ===============================

strength_threshold = roi_df["Structural_Strength"].median()
gain_threshold = roi_df["Raw_Gain"].median()

def categorize(row):
    
    if row["Raw_Gain"] > gain_threshold and row["Structural_Strength"] < strength_threshold:
        return "Breakout Potential"
    
    elif row["Raw_Gain"] > gain_threshold and row["Structural_Strength"] >= strength_threshold:
        return "High-Return Investment"
    
    elif row["Raw_Gain"] <= gain_threshold and row["Structural_Strength"] >= strength_threshold:
        return "Stable Core Sport"
    
    else:
        return "Low Priority"

roi_df["Strategy"] = roi_df.apply(categorize, axis=1)

# ===============================
# DISPLAY OUTPUT
# ===============================

print("\n--- Strategic Sport Intelligence (Stabilized) ---\n")
display(
    roi_df[
        ["Sport", "Base_Prediction", "Raw_Gain", "ROI_Percentage", "Strategy"]
    ].round(3)
)


--- Strategic Sport Intelligence (Stabilized) ---



,Sport,Base_Prediction,Raw_Gain,ROI_Percentage,Strategy
14,Hockey,0.166,0.953,381.047,High-Return Investment
0,Alpine Skiing,0.006,0.953,381.047,Stable Core Sport
11,Football,0.006,0.953,381.047,Stable Core Sport
22,Water Polo,0.006,0.953,381.047,Stable Core Sport
20,Table Tennis,0.006,0.953,381.047,Stable Core Sport
19,Swimming,0.006,0.953,381.047,Stable Core Sport
17,Sailing,0.006,0.953,381.047,Stable Core Sport
16,Rowing,0.006,0.953,381.047,Stable Core Sport
15,Judo,0.006,0.953,381.047,Stable Core Sport
13,Gymnastics,0.006,0.953,381.047,Stable Core Sport


In [13]:
# ==========================================
# IMPROVISED BREAKOUT DETECTOR (STABLE)
# ==========================================

epsilon = 0.25 
latest["Breakout_Intensity"] = latest["delta_last"] / (latest["career_avg"] + epsilon)
latest["Breakout_Impact"] = latest["Breakout_Intensity"] * latest["Predicted_Medals_2028"]

# Use a more flexible filter to ensure the list is never empty
median_career = latest["career_avg"].median()
# We look for sports with positive momentum OR high intensity
breakout_candidates = latest[
    (latest["delta_last"] >= 0) & 
    (latest["career_avg"] <= median_career)
].copy()

if not breakout_candidates.empty:
    # Normalize the Impact into a 0-100 "Heat Index"
    max_impact = breakout_candidates["Breakout_Impact"].max()
    min_impact = breakout_candidates["Breakout_Impact"].min()
    
    if max_impact != min_impact:
        breakout_candidates["Breakout_Index"] = (
            (breakout_candidates["Breakout_Impact"] - min_impact) / (max_impact - min_impact) * 100
        ).round(1)
    else:
        breakout_candidates["Breakout_Index"] = 50.0  # Baseline if all are equal

    breakout_candidates = breakout_candidates.sort_values("Breakout_Index", ascending=False)

    print("\n--- 💎 Top Breakout & Emerging Candidates for India 2028 ---\n")
    display(
        breakout_candidates[
            ["Sport", "career_avg", "delta_last", "Predicted_Medals_2028", "Breakout_Index"]
        ]
    )
else:
    print("\n--- No clear breakout candidates found with current thresholds. ---")


--- 💎 Top Breakout & Emerging Candidates for India 2028 ---



,Sport,career_avg,delta_last,Predicted_Medals_2028,Breakout_Index
645,Alpine Skiing,0.0,0.0,0.00607,50.0
639,Archery,0.0,0.0,0.00607,50.0
646,Art Competitions,0.0,0.0,0.00607,50.0
647,Basketball,0.0,0.0,0.00607,50.0
634,Cycling,0.0,0.0,0.00607,50.0
648,Diving,0.0,0.0,0.00607,50.0
630,Equestrianism,0.0,0.0,0.00607,50.0
631,Football,0.0,0.0,0.00607,50.0
636,Golf,0.0,0.0,0.00607,50.0
644,Gymnastics,0.0,0.0,0.00607,50.0


In [14]:
import pickle, os
os.makedirs("../models", exist_ok=True)
with open("../models/india_sport_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("✅ Sport model saved")


✅ Sport model saved


In [15]:
os.makedirs("../data/processed", exist_ok=True)

# Main sport predictions
latest[["Sport","career_avg","delta_last",
        "Predicted_Medals_2028","Scaled_Pred_2028"]].to_csv(
    "../data/processed/india_sport_predictions_2028.csv", index=False)

# Sensitivity matrix
sensitivity_df.to_csv(
    "../data/processed/india_sport_sensitivity.csv", index=False)

# ROI table
roi_df.to_csv(
    "../data/processed/india_sport_roi.csv", index=False)

# Breakout candidates
if not breakout_candidates.empty:
    breakout_candidates[["Sport","career_avg","delta_last",
                          "Predicted_Medals_2028","Breakout_Index"]].to_csv(
        "../data/processed/india_sport_breakout.csv", index=False)

print("✅ All sport intelligence CSVs exported")

✅ All sport intelligence CSVs exported
